# Data Preparation: Feature Engineering

This notebook mirrors the **Data Preparation** guide at [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/).

Transform the validated employee attrition dataset into a model-ready feature matrix: encode categoricals, aggregate satisfaction signals, bin continuous variables, and engineer risk-indicator features.

**What we'll cover:**
1. Load validated data
2. Define the feature engineering function
3. Apply feature engineering
4. Inspect the result and write `featured.csv`

In [ ]:
import pandas as pd
import numpy as np

## Step 1: Load validated data

In [ ]:
df = pd.read_csv('/workspace/pipeline-data/validated.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## Step 2: Define the feature engineering function

In [ ]:
def feature_data(df):
    df_fe = df.copy()

    # --- Encoding ---
    # Target column: Yes → 1, No → 0
    df_fe['Attrition'] = df_fe['Attrition'].map({'Yes': 1, 'No': 0}).astype('int')

    # Binary encoding
    df_fe['OverTime'] = df_fe['OverTime'].map({'Yes': 1, 'No': 0}).astype('Int64')
    df_fe['Gender'] = df_fe['Gender'].map({'Male': 1, 'Female': 0}).astype('Int64')

    # Ordinal encoding
    df_fe['BusinessTravel'] = df_fe['BusinessTravel'].map(
        {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
    ).astype('Int64')
    df_fe['MaritalStatus'] = df_fe['MaritalStatus'].map(
        {'Single': 0, 'Married': 1, 'Divorced': 2}
    ).astype('Int64')

    # --- Aggregate satisfaction signals into a single score ---
    # WorkLifeBalance, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction
    # are already numeric (1–4); no encoding needed before aggregating
    df_fe['OverallSatisfaction'] = (
        (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] +
         df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
    ).round().astype('Int64')
    df_fe = df_fe.drop(columns=['JobSatisfaction', 'EnvironmentSatisfaction',
                                 'RelationshipSatisfaction', 'WorkLifeBalance'])

    # --- Bin continuous variables ---
    # Annual income bands (monthly × 12)
    df_fe['AnnualIncome'] = pd.cut(
        df_fe['MonthlyIncome'] * 12,
        bins=[0, 25000, 50000, 75000, 100000, float('inf')],
        labels=[0, 1, 2, 3, 4],
        include_lowest=True
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['MonthlyIncome'])

    # Age bands
    df_fe['AgeGroup'] = pd.cut(
        df_fe['Age'],
        bins=[17, 30, 40, 50, 65],
        labels=[1, 2, 3, 4]
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['Age'])

    # --- Derived risk-indicator features ---
    # High ratio → employee stuck without promotion
    df_fe['PromotionStagnation'] = (
        df_fe['YearsSinceLastPromotion'] / (df_fe['YearsAtCompany'] + 1)
    ).round(3)

    # Job level earned relative to total experience
    df_fe['CareerVelocity'] = (
        df_fe['JobLevel'] / (df_fe['TotalWorkingYears'] + 1)
    ).round(3)

    # Commute distance risk indicator
    df_fe['LongCommute'] = (df_fe['DistanceFromHome'] > 20).astype('Int64')

    # Stable manager relationship (reduces attrition risk)
    df_fe['StableManager'] = (df_fe['YearsWithCurrManager'] >= 3).astype('Int64')

    # Drop ID columns, constants, and high-cardinality categoricals
    drop_cols = [c for c in [
        'EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18',
        'JobRole', 'EducationField', 'Department',
        'DistanceFromHome', 'YearsWithCurrManager',
    ] if c in df_fe.columns]
    df_fe = df_fe.drop(columns=drop_cols)

    return df_fe

The function applies transformations in four logical groups:
1. **Encoding** — convert string categories to numeric (binary and ordinal)
2. **Aggregation** — collapse four satisfaction scores into `OverallSatisfaction`
3. **Binning** — convert `MonthlyIncome` and `Age` into ordinal bands
4. **Derived features** — engineer signals that correlate with attrition (`PromotionStagnation`, `CareerVelocity`, `LongCommute`, `StableManager`)

## Step 3: Apply feature engineering

In [ ]:
df_featured = feature_data(df)
print(f'Output shape: {df_featured.shape}')
print(f'Columns: {list(df_featured.columns)}')

## Step 4: Inspect the result

In [ ]:
print('Dtypes:')
print(df_featured.dtypes)
print()
df_featured.head()

In [ ]:
output_path = '/workspace/pipeline-data/featured.csv'
df_featured.to_csv(output_path, index=False)
print(f'Written: {output_path}')

## Next Steps

- Continue to preprocessing and train/test split: `02-pipeline/data-preparation/preprocess.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/data-preparation/](https://learnmlops.ops4life.com/guides/data-preparation/)